# 实践项目 02：胸片 DCGAN 图像生成

我们将在 Kaggle Notebook 中读取真实胸片，训练轻量 DCGAN，观察生成器与判别器的训练变化，并比较训练早期和后期的生成样本。

Kaggle 是本项目的首选实践入口。打开公开 Notebook 后，先点击“复制并编辑”保存到自己的账户，再逐步运行、修改代码并观察自己的输出；下载 Notebook 到电脑运行是补充方式。
代码填写位置：标有 TODO，或明确写成 pass、None 占位的代码位置，以及标有“你的回答”的 Markdown 单元格，就是需要完成的部分。先阅读当前任务说明，再根据变量名、输入输出 shape、注释和下一步的 print/assert 填写；不要直接把参考结果数字写进代码。
完成一个任务后，先运行当前单元格和后续检查单元格，确认输出形状、指标和输出文件符合说明，再进入下一项。

## 实践任务
1. 读取 NORMAL 与 PNEUMONIA 目录中的胸片
2. 完成灰度化、缩放和 [-1,1] 归一化
3. 生成真实胸片网格和像素分布图
4. 补全生成器的一段网络结构
5. 完成判别器与生成器的交替训练步骤
6. 使用固定噪声保存不同轮次的生成样本
7. 结合损失曲线检查训练平衡与样本多样性

## 需要保存的结果
- `task2_real_xray_grid.png`
- `task2_intensity_histogram.png`
- `task2_generated_samples.png`
- `task2_training_curve.png`
- `task2_pytorch_result.json`


## 输出
- `task2_real_xray_grid.png`：真实胸片样本网格
- `task2_intensity_histogram.png`：真实胸片像素强度分布
- `task2_generated_samples.png`：固定噪声生成样本网格
- `task2_pytorch_result.json`：训练设置与结果记录
- `task2_training_curve.png`：生成器与判别器损失曲线（补充输出）


In [ ]:
from pathlib import Path
import json, random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader,Subset
from torchvision import datasets,transforms,utils

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
cuda_ok=torch.cuda.is_available()
if cuda_ok:
    try:
        cuda_ok=torch.cuda.get_device_capability()[0]>=7
    except Exception:
        cuda_ok=False
DEVICE=torch.device('cuda' if cuda_ok else 'cpu')
OUT=Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()/'outputs'; OUT.mkdir(exist_ok=True)
print(DEVICE)

## 1. 数据路径与预处理

推荐挂载 `chest-xray-pneumonia`。本实践使用训练集中 NORMAL 与 PNEUMONIA 图像共同学习胸片外观，不把目录标签输入生成器。下载到电脑运行时，把数据整理为 `data/train/NORMAL` 和 `data/train/PNEUMONIA`，输出会保存到 `outputs` 文件夹。


In [ ]:
INPUT_ROOT=Path('/kaggle/input') if Path('/kaggle/input').exists() else Path.cwd()/'data'
candidates=[p for p in INPUT_ROOT.glob('**/train') if (p/'NORMAL').exists() or (p/'PNEUMONIA').exists()]
assert candidates,'未找到胸片 train 目录。'
DATA_ROOT=candidates[0]
transform=transforms.Compose([transforms.Grayscale(1),transforms.Resize(72),transforms.CenterCrop(64),transforms.ToTensor(),transforms.Normalize([.5],[.5])])
ds=datasets.ImageFolder(DATA_ROOT,transform=transform)
MAX_IMAGES=min(1000,len(ds)); ds=Subset(ds,list(range(MAX_IMAGES)))
NUM_WORKERS=2 if Path('/kaggle/input').exists() else 0
loader=DataLoader(ds,batch_size=64,shuffle=True,num_workers=NUM_WORKERS,drop_last=True)
real,_=next(iter(loader)); utils.save_image((real[:36]+1)/2,OUT/'task2_real_xray_grid.png',nrow=6)

# 保存真实胸片的像素强度分布。先映射回 [0,1]，再绘制直方图。
real_display=((real+1)/2).clamp(0,1)
plt.figure(figsize=(7.2,4.2))
plt.hist(real_display.flatten().numpy(),bins=60,color='#2b7b9b',alpha=.88)
plt.xlabel('pixel intensity [0,1]'); plt.ylabel('count')
plt.title('Real chest X-ray intensity distribution')
plt.tight_layout(); plt.savefig(OUT/'task2_intensity_histogram.png',dpi=160); plt.show()
print('images:',len(ds),real.shape,real.min().item(),real.max().item())

## 任务 1：解释归一化范围

说明为什么真实图像被转换到 [-1,1]，以及生成器最后一层应使用什么激活函数。把答案写在下方 Markdown 单元。


**你的回答：**


## 任务 2：补全生成器


In [ ]:
LATENT=100
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO 2：从 [B,100,1,1] 上采样到 [B,1,64,64]
        self.net=None
    def forward(self,z): return self.net(z)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(1,32,4,2,1),nn.LeakyReLU(.2,True),
            nn.Conv2d(32,64,4,2,1),nn.BatchNorm2d(64),nn.LeakyReLU(.2,True),
            nn.Conv2d(64,128,4,2,1),nn.BatchNorm2d(128),nn.LeakyReLU(.2,True),
            nn.Conv2d(128,256,4,2,1),nn.BatchNorm2d(256),nn.LeakyReLU(.2,True),
            nn.Conv2d(256,1,4,1,0))
    def forward(self,x): return self.net(x).flatten()

G=Generator().to(DEVICE); D=Discriminator().to(DEVICE)
print(G(torch.randn(2,LATENT,1,1,device=DEVICE)).shape)

## 任务 3：补全对抗训练步骤


In [ ]:
criterion=nn.BCEWithLogitsLoss()
opt_g=torch.optim.Adam(G.parameters(),lr=2e-4,betas=(.5,.999))
opt_d=torch.optim.Adam(D.parameters(),lr=2e-4,betas=(.5,.999))
fixed_z=torch.randn(36,LATENT,1,1,device=DEVICE)
history={'g':[],'d':[]}

for epoch in range(4):
    for real,_ in loader:
        real=real.to(DEVICE); b=len(real)
        ones=torch.ones(b,device=DEVICE); zeros=torch.zeros(b,device=DEVICE)
        # TODO 3A：更新判别器。生成图像在此阶段需要 detach
        loss_d=None
        # TODO 3B：更新生成器，使生成图像被判为真实
        loss_g=None
        history['d'].append(float(loss_d)); history['g'].append(float(loss_g))
    with torch.no_grad(): fake=G(fixed_z).cpu()
    utils.save_image((fake+1)/2,OUT/f'task2_epoch_{epoch+1}.png',nrow=6)
    print(epoch+1,np.mean(history['d'][-len(loader):]),np.mean(history['g'][-len(loader):]))

In [ ]:
plt.figure(figsize=(8,3.5)); plt.plot(history['d'],label='discriminator'); plt.plot(history['g'],label='generator'); plt.legend(); plt.xlabel('batch'); plt.ylabel('loss'); plt.tight_layout(); plt.savefig(OUT/'task2_training_curve.png',dpi=160); plt.show()
with torch.no_grad(): generated=G(fixed_z).cpu()
utils.save_image((generated+1)/2,OUT/'task2_generated_samples.png',nrow=6)

flat=generated.flatten(1)
dist=torch.cdist(flat,flat); diversity=float(dist[torch.triu(torch.ones_like(dist),diagonal=1).bool()].mean())
result={'dataset_images':len(ds),'epochs':4,'latent_dim':LATENT,'mean_generator_loss_last_epoch':float(np.mean(history['g'][-len(loader):])),'mean_discriminator_loss_last_epoch':float(np.mean(history['d'][-len(loader):])),'generated_pairwise_distance':diversity,'seed':SEED}
(OUT/'task2_pytorch_result.json').write_text(json.dumps(result,indent=2),encoding='utf-8'); result

## 任务 4：结果检查

从生成网格中记录三类现象：结构较完整的部分、明显伪影和重复样本。结合两条损失曲线与像素直方图，写下结构、伪影、重复样本、损失和距离的观察。

**你的回答：**

- 结构观察：
- 伪影观察：
- 重复样本与多样性：
- 损失曲线和像素分布：
- 综合判断：
